# Radar ECOBICI CDMX — Análisis exploratorio (EDA)

| | |
|---|---|
| **Autor** | Isacc Cabrera |
| **Descripción** | Exploración del lake (silver, gold, clima y viajes históricos) antes de entrenar el modelo t+60. |
| **Patrón** | N/A (notebook) |
| **Fecha de creación** | 2026-11-10 |
| **Cambios** | 2026-11-10 · Isacc Cabrera · Creación (paso 10). |

## Introducción

ECOBICI publica cada minuto cuántas bicis y muelles libres tiene cada una de sus ~677 estaciones
(feed GBFS). Radar ECOBICI guarda ese feed en un data lake (bronze → silver → gold) y quiere
predecir **cuántas bicis habrá en cada estación dentro de 60 minutos**. Este notebook revisa
qué tan confiables son los datos y qué patrones tienen antes de construir el modelo.

## Objetivos

1. Medir la **calidad** de lo recolectado: cobertura del recolector (heartbeat), cumplimiento de
   la regla de capacidad y cuarentena.
2. Ver **dónde** están las estaciones vacías o llenas (mapa).
3. Entender la **dinámica temporal**: qué estaciones cambian más y cómo varía la red por hora y día.
4. Revisar si el **clima** se relaciona con el uso.
5. Dejar conclusiones que guíen las features y el modelo (paso 11).

In [ ]:
from pathlib import Path
import json

import pandas as pd
import plotly.express as px
import plotly.io as pio

# El kernel arranca en notebooks/: subimos a la raíz del repo si ahí está pyproject.toml
ROOT = Path.cwd().parent if (Path.cwd().parent / "pyproject.toml").exists() else Path.cwd()
DATA = ROOT / "data"
pio.renderers.default = "notebook_connected"  # plotly.js desde CDN: HTML ligero
pd.set_option("display.max_columns", 20)
TZ = "America/Mexico_City"
print("Raíz del repo:", ROOT)

## Datos

| Capa | Tabla | Qué contiene |
|---|---|---|
| bronze | `heartbeat/dt=*.jsonl` | una línea por corrida del recolector (cada 60 s) |
| silver | `station_status` | un renglón por cambio de estado de cada estación, validado |
| silver | `quarantine` | filas rechazadas por la cadena de calidad, con su motivo |
| silver | `station_information` | nombre, coordenadas y capacidad de cada estación |
| silver | `weather` | clima horario de Open-Meteo (temperatura, lluvia) |
| gold | `features_base` | rejilla de 5 min por estación con el último estado conocido |
| gold | `station_hour` | agregados por estación y hora |
| gold | `trip_demand` | salidas/llegadas promedio por estación, día y hora (viajes de octubre, sin datos personales) |

Todos los tiempos del lake están en UTC; aquí se convierten a hora de CDMX para leerlos.

In [ ]:
info = pd.read_parquet(DATA / "silver/station_information/station_information.parquet")
base = pd.read_parquet(DATA / "gold/features_base")
hour = pd.read_parquet(DATA / "gold/station_hour")
weather = pd.read_parquet(DATA / "silver/weather/weather.parquet")
demand = pd.read_parquet(DATA / "gold/trip_demand/trip_demand.parquet")
for df in (base, hour):
    df["station_id"] = df["station_id"].astype(str)
base["ts_local"] = base["ts_utc"].dt.tz_convert(TZ)
base["pct_bicis"] = base["bikes_available"] / base["capacity"].where(base["capacity"] > 0)

resumen = pd.DataFrame(
    {
        "filas": [len(info), len(base), len(hour), len(weather), len(demand)],
        "desde": [None, base.ts_utc.min(), hour.hora_utc.min(), weather.ts_utc.min(), None],
        "hasta": [None, base.ts_utc.max(), hour.hora_utc.max(), weather.ts_utc.max(), None],
    },
    index=["station_information", "features_base", "station_hour", "weather", "trip_demand"],
)
print("Estaciones:", info.station_id.nunique(), "· horas con datos en gold:", base.ts_utc.dt.floor("h").nunique())
resumen

## Calidad

Tres preguntas: ¿el recolector corrió sin huecos?, ¿los datos cumplen la regla de capacidad
(`bicis + bicis dañadas + muelles + muelles dañados == capacidad`)? y ¿qué se fue a cuarentena?

In [ ]:
rows = []
for path in sorted((DATA / "bronze/heartbeat").glob("dt=*.jsonl")):
    for line in path.read_text(encoding="utf-8").splitlines():
        if line.strip():
            rows.append(json.loads(line))
hb = pd.DataFrame(rows)
hb = hb[hb["feed"] == "station_status"].copy()
hb["ts"] = pd.to_datetime(hb["ts_utc"], utc=True, format="ISO8601")
hb["hora"] = hb["ts"].dt.floor("h")
cobertura = (
    hb[hb["result"].isin(["guardado", "sin_cambios"])].groupby("hora").size().rename("corridas_ok")
)
todas = pd.date_range(hb["hora"].min(), hb["hora"].max(), freq="h", tz="UTC")
cobertura = cobertura.reindex(todas, fill_value=0)
print(f"Horas observadas: {len(todas)} · completas (≥ 55 corridas): {(cobertura >= 55).sum()} "
      f"· sin ninguna corrida: {(cobertura == 0).sum()}")
print("Resultados del heartbeat:", hb["result"].value_counts().to_dict())
fig = px.bar(
    x=cobertura.index.tz_convert(TZ), y=cobertura.values,
    labels={"x": "Hora (CDMX)", "y": "Corridas exitosas"},
    title="Cobertura del recolector por hora (60 = sin huecos)",
)
fig.add_hline(y=60, line_dash="dot")
fig.show()

In [ ]:
manifiestos = []
for path in sorted((DATA / "silver/station_status").glob("dt=*/hour=*/_manifest.json")):
    manifiestos.append(json.loads(path.read_text(encoding="utf-8")))
man = pd.DataFrame(manifiestos)
man["pct_regla"] = man["tasa_regla_capacidad"] * 100
print(f"Tasa de la regla de capacidad: {man.pct_regla.min():.2f} % – {man.pct_regla.max():.2f} %")
print(f"Filas leídas: {man.filas_leidas.sum():,} · duplicadas: {man.filas_duplicadas.sum():,} "
      f"({man.filas_duplicadas.sum() / man.filas_leidas.sum():.0%}) · cuarentena: {man.filas_cuarentena.sum():,}")
quarantine = pd.read_parquet(DATA / "silver/quarantine")
q = quarantine.groupby("motivo").agg(filas=("station_id", "size"), estaciones=("station_id", "nunique"))
q

In [ ]:
peores = (
    quarantine.assign(station_id=quarantine["station_id"].astype(str))
    .groupby("station_id").size().rename("filas_cuarentena").reset_index()
    .merge(info[["station_id", "short_name", "name", "capacity"]], on="station_id")
    .sort_values("filas_cuarentena", ascending=False)
)
peores.head(11)

**Horas "congeladas".** La rejilla de gold toma el último valor conocido. Si en una hora el
recolector corrió solo unos minutos, el resto de la hora repite ese valor y parece que nada se
movió. Se mide con el % de puntos de la rejilla cuyo valor cambió respecto al punto anterior,
junto a las corridas del heartbeat de esa hora.

In [ ]:
b = base.sort_values(["station_id", "ts_utc"])
b["cambio"] = b.groupby("station_id")["bikes_available"].diff().fillna(0) != 0
congeladas = (
    b.groupby(b["ts_utc"].dt.floor("h"))["cambio"].mean().mul(100).round(1).rename("pct_puntos_con_cambio")
    .to_frame().join(cobertura.rename("corridas_ok"))
)
congeladas.index = congeladas.index.tz_convert(TZ)
congeladas

## Mapa

Cada punto es una estación, coloreada por el % de bicis disponibles respecto a su capacidad en
el último instante de la rejilla (rojo = vacía, verde = llena). Mapa base libre, sin token.

In [ ]:
ultimo = base[base["ts_utc"] == base["ts_utc"].max()]
mapa = ultimo.merge(info[["station_id", "short_name", "name", "lat", "lon"]], on="station_id")
mapa["pct"] = (mapa["pct_bicis"] * 100).round(1)
print("Instante:", ultimo["ts_local"].iloc[0], "· vacías:", (mapa.bikes_available == 0).sum(),
      "· llenas:", (mapa.docks_available == 0).sum(), "de", len(mapa))
fig = px.scatter_map(
    mapa, lat="lat", lon="lon", color="pct", color_continuous_scale="RdYlGn", range_color=(0, 100),
    hover_name="name", hover_data={"short_name": True, "bikes_available": True, "capacity": True,
                                   "lat": False, "lon": False},
    zoom=11.3, height=560, map_style="open-street-map",
    title="% de bicis disponibles por estación",
)
fig.show()

## Series de tiempo

Las 5 estaciones con mayor desviación estándar de bicis disponibles en la rejilla: son las que
más se mueven y las más difíciles de predecir. Los tramos sin línea son horas sin recolector.

In [ ]:
variab = base.groupby("station_id")["bikes_available"].std().sort_values(ascending=False)
top5 = variab.head(5).index.tolist()
serie = base[base["station_id"].isin(top5)].merge(info[["station_id", "short_name", "name"]], on="station_id")
serie["estacion"] = serie["short_name"] + " · " + serie["name"].str.slice(0, 28)
# cortar la línea en los huecos: reindexar cada estación a la rejilla completa
grid = pd.date_range(base.ts_utc.min(), base.ts_utc.max(), freq="5min", tz="UTC")
partes = []
for sid, g in serie.groupby("station_id"):
    g = g.set_index("ts_utc").reindex(grid)
    g["estacion"] = g["estacion"].ffill().bfill()
    partes.append(g.reset_index(names="ts_utc"))
serie = pd.concat(partes)
serie["ts_local"] = serie["ts_utc"].dt.tz_convert(TZ)
fig = px.line(serie, x="ts_local", y="bikes_available", color="estacion",
              labels={"ts_local": "Hora (CDMX)", "bikes_available": "Bicis disponibles"},
              title="Las 5 estaciones más variables")
fig.show()
variab.head(5).round(2).rename("desv_std_bicis").to_frame().join(info.set_index("station_id")[["short_name", "name", "capacity"]])

## Patrón por hora y día

Con pocas horas de feed, el patrón semanal sale de los **viajes históricos de octubre**
(`trip_demand`): salidas promedio de toda la red por día de la semana y hora local. El feed
recolectado muestra el mismo patrón visto desde las estaciones: % de estaciones vacías por hora.

In [ ]:
dias = ["lun", "mar", "mié", "jue", "vie", "sáb", "dom"]
red = demand.groupby(["dow", "hour_local"])[["salidas_prom", "llegadas_prom"]].sum().reset_index()
tabla = red.pivot(index="dow", columns="hour_local", values="salidas_prom").reindex(range(7))
tabla.index = dias
fig = px.imshow(tabla, aspect="auto", color_continuous_scale="Greens",
                labels={"x": "Hora local", "y": "Día", "color": "Salidas/h"},
                title="Salidas promedio por hora en toda la red (viajes de octubre de 2026)")
fig.show()
pico = red.loc[red.salidas_prom.idxmax()]
print(f"Hora pico: {dias[int(pico.dow)]} a las {int(pico.hour_local)} h con {pico.salidas_prom:,.0f} salidas promedio")
habil = red[red.dow < 5].groupby("hour_local").salidas_prom.mean()
finde = red[red.dow >= 5].groupby("hour_local").salidas_prom.mean()
print(f"Día hábil: pico a las {habil.idxmax()} h ({habil.max():,.0f}); fin de semana: pico a las {finde.idxmax()} h ({finde.max():,.0f})")

In [ ]:
base["hora_local"] = base["ts_local"].dt.floor("h")
porhora = base.groupby("hora_local").agg(
    pct_vacias=("bikes_available", lambda s: (s == 0).mean() * 100),
    pct_llenas=("docks_available", lambda s: (s == 0).mean() * 100),
    bicis_prom=("bikes_available", "mean"),
).reset_index()
fig = px.line(porhora.melt(id_vars="hora_local", value_vars=["pct_vacias", "pct_llenas"]),
              x="hora_local", y="value", color="variable", markers=True,
              labels={"hora_local": "Hora (CDMX)", "value": "% de estaciones"},
              title="Estaciones vacías y llenas por hora (feed recolectado)")
fig.show()
porhora.round({"pct_vacias": 1, "pct_llenas": 1, "bicis_prom": 2})

## Clima vs uso

Temperatura y lluvia de Open-Meteo frente a las bicis disponibles en toda la red (menos bicis
estacionadas ≈ más bicis en uso). Con pocas horas recolectadas es una primera mirada, no una
conclusión estadística.

In [ ]:
w = weather.copy()
w["hora"] = w["ts_utc"]
# total de la red en cada punto de la rejilla y luego el promedio de la hora (horas parciales no se subestiman)
por_punto = base.groupby("ts_utc")["bikes_available"].sum()
uso = por_punto.groupby(por_punto.index.floor("h")).mean().rename("bicis_estacionadas").round(0)
uso = uso.rename_axis("hora").reset_index().merge(w, on="hora", how="inner")
uso["hora_local"] = uso["hora"].dt.tz_convert(TZ).dt.hour
fig = px.scatter(uso, x="temperature_2m", y="bicis_estacionadas", color="hora_local",
                 size=uso["precipitation"] + 0.5,
                 labels={"temperature_2m": "Temperatura (°C)", "bicis_estacionadas": "Bicis estacionadas en la red"},
                 title="Clima vs bicis estacionadas (una burbuja por hora; tamaño = lluvia)")
fig.show()
print("Correlación temperatura–bicis estacionadas:", round(uso["temperature_2m"].corr(uso["bicis_estacionadas"]), 2),
      f"(n = {len(uso)} horas)")
uso[["hora", "hora_local", "temperature_2m", "precipitation", "bicis_estacionadas"]]

In [ ]:
from radar.ml.features import FEATURES, build_training_frame, demand_by_station

train = build_training_frame(base, weather, demand_by_station(demand, info))
print(f"Marco de entrenamiento: {len(train):,} filas × {len(FEATURES)} features · estaciones: {train.station_id.nunique()}")
print("Persistencia (y_t60 = bicis ahora): MAE =",
      round((train["y_t60"] - train["bikes_available"]).abs().mean(), 3))
print("Faltantes por feature (%):")
(train[FEATURES].isna().mean() * 100).round(1)[lambda s: s > 0]

## Conclusiones

Observaciones con los datos al 10 de noviembre de 2026 (11 horas de feed en gold, 677
estaciones; viajes de octubre de 2026 para el patrón semanal). Las cifras de las celdas de arriba
se recalculan cada vez que se ejecuta el notebook.

1. **Calidad: los datos son confiables, pero la cobertura tiene huecos.**
   - La regla de capacidad se cumple en ~99.7 % de las filas. La cuarentena se concentra en unas
     13 estaciones (la CE-028 Toledo-Tokio sola aporta más del 40 %), siempre por ±1: es un
     problema de esas estaciones, no del pipeline.
   - ~78 % de lo que llega del feed son repeticiones (minutos sin cambios); silver las descarta.
   - Hay horas sin recolector (la laptop suspendida) y dos horas **"congeladas"** (04 y 12 UTC)
     con solo 3–4 corridas: la rejilla repite un valor viejo toda la hora. **Para entrenar
     (paso 11) se deben excluir las horas con cobertura incompleta**, o el modelo aprenderá que
     "nada cambia".
2. **Dónde está el problema.** A las 07:55 del martes, 129 de 677 estaciones estaban vacías y 19
   llenas. Las estaciones compuestas de Buenavista (`264-275`, `268-269`, `266-267`, `273-274`,
   `271-272`, capacidad ~70) son las más variables: se vacían por la mañana con los viajeros que
   llegan en tren. Son el caso más difícil y más útil de predecir.
3. **El patrón diario es fuerte.** En día hábil la red tiene su pico de salidas a las 8 h y otro
   por la tarde (máximo del mes: martes 17 h); en fin de semana el pico se mueve al mediodía
   (13 h). En el feed, las estaciones vacías pasan de ~14 % a las 14 h a ~32 % a las 19 h.
   ⇒ `hour_local`, `dow`, `is_weekend` y la demanda histórica son features con sentido.
4. **Clima.** Con 11 horas la relación temperatura–uso se confunde con la hora del día (hace más
   frío de noche, cuando también hay menos viajes). No hay base para una conclusión; el clima
   entra como feature y el modelo dirá si aporta.
5. **Punto de partida del modelo.** Predecir "lo mismo que ahora" (persistencia) tiene un MAE de
   ~3 bicis a 60 minutos. **Ese es el número a vencer** en el paso 11; los lags y ventanas solo
   miran el pasado y la validación será temporal (nunca al azar).
